In [1]:
import pickle
import plotly.graph_objects as go
import os
import numpy as np
from GridMLM_tokenizers import CSGridMLMTokenizer

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
with open('data/gjt_train_harmony_ids.pickle', 'rb') as handle:
    gjt_train_harmony_ids = pickle.load(handle)
with open('data/gjt_val_harmony_ids.pickle', 'rb') as handle:
    gjt_val_harmony_ids = pickle.load(handle)

with open('data/nott_train_harmony_ids.pickle', 'rb') as handle:
    nott_train_harmony_ids = pickle.load(handle)
with open('data/nott_val_harmony_ids.pickle', 'rb') as handle:
    nott_val_harmony_ids = pickle.load(handle)

In [4]:
with open('data/umap/umap_2d_per_layer.pickle', 'rb') as handle:
    umap_2d_per_layer = pickle.load(handle)

In [5]:
print(umap_2d_per_layer.keys())

dict_keys([0, 1, 2, 3, 4, 5, 6, 7])


In [6]:
def make_interactive_plot_for_layer(k):
    fig = go.Figure()

    gjt_train_2d = umap_2d_per_layer[k]['gjt_train_2d']
    gjt_val_2d = umap_2d_per_layer[k]['gjt_val_2d']
    nott_train_2d = umap_2d_per_layer[k]['nott_train_2d']
    nott_val_2d = umap_2d_per_layer[k]['nott_val_2d']

    groups = [
        ("GJT train", gjt_train_2d, gjt_train_harmony_ids, "firebrick"),
        ("GJT validation", gjt_val_2d, gjt_val_harmony_ids, "fuchsia"),
        ("NoTT train", nott_train_2d, nott_train_harmony_ids, "blue"),
        ("NoTT validation", nott_val_2d, nott_val_harmony_ids, "cyan"),
    ]

    for name, coords, harmony_ids, color in groups:
        harmony_ids = np.asarray(harmony_ids).reshape(-1)
        harmony_tokens = np.asarray(
            [tokenizer.ids_to_tokens[int(token_id)] for token_id in harmony_ids]
        ).reshape(-1, 1)
        assert len(coords) == len(harmony_ids), f"{name}: coordinates and IDs differ in length"

        fig.add_trace(go.Scattergl(
            x=coords[:, 0],
            y=coords[:, 1],
            mode="markers",
            name=name,
            customdata=harmony_tokens,
            # customdata=harmony_ids.reshape(-1, 1),
            marker=dict(color=color, size=5, opacity=0.35),
            hovertemplate=(
                f"{name}<br>"
                "Harmony ID: %{customdata[0]}<br>"
                "UMAP 1: %{x:.3f}<br>"
                "UMAP 2: %{y:.3f}"
                "<extra></extra>"
            ),
        ))

    fig.update_layout(
        title=f"UMAP of latent layer {k}",
        xaxis_title="Component 1",
        yaxis_title="Component 2",
        hovermode="closest",
    )

    os.makedirs('html/umap', exist_ok=True)
    fig.write_html(f"html/umap/umap_{k}.html", include_plotlyjs="cdn")
    fig.show()

In [14]:
make_interactive_plot_for_layer(7)